# Fine-Tuning Mathematics — Exercises

**8 hands-on exercises** covering LoRA, DPO, QLoRA, EWC, adapters, IFD, GRPO, and model merging.

Each exercise has:
1. **Problem statement** with concrete numbers
2. **Scaffold** code to fill in
3. **Solution** with step-by-step working

[← Theory Notebook](theory.ipynb) | [Notes](notes.md)

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def print_vector(v, fmt=".4f"):
    return "[" + ", ".join(f"{x:{fmt}}" for x in v) + "]"

print("Setup complete ✓")

## Exercise 1: LoRA Parameter Count

**Problem:** A model has $d = 4096$, rank $r = 16$, LoRA applied to **6 matrices** (Q, K, V, O, up, down) across $L = 32$ layers. The full model has 8B parameters.

Calculate:
1. LoRA parameters **per matrix**
2. LoRA parameters **per layer**
3. **Total** LoRA parameters
4. **Percentage** of full model

In [ ]:
# Exercise 1: Scaffold — fill in the ???

d = 4096
r = 16
n_matrices = 6
L = 32
total_model_params = 8e9

# Per matrix: A ∈ R^(r×d) + B ∈ R^(d×r)
params_per_matrix = None  # ??? = r * d + d * r = 2 * r * d

# Per layer
params_per_layer = None   # ??? = n_matrices * params_per_matrix

# Total
total_lora = None         # ??? = L * params_per_layer

# Percentage
percentage = None         # ??? = total_lora / total_model_params * 100

print(f"Per matrix:  {params_per_matrix:,}")
print(f"Per layer:   {params_per_layer:,}")
print(f"Total LoRA:  {total_lora:,.0f}")
print(f"Percentage:  {percentage:.3f}%")

In [ ]:
# Exercise 1: SOLUTION

d = 4096
r = 16
n_matrices = 6
L = 32
total_model_params = 8e9

params_per_matrix = 2 * r * d           # A: r×d, B: d×r
params_per_layer = n_matrices * params_per_matrix
total_lora = L * params_per_layer
percentage = total_lora / total_model_params * 100

print("=== Exercise 1 Solution ===\n")
print(f"Per matrix:  2 × {r} × {d} = {params_per_matrix:,}")
print(f"Per layer:   {n_matrices} × {params_per_matrix:,} = {params_per_layer:,}")
print(f"Total LoRA:  {L} × {params_per_layer:,} = {total_lora:,.0f}")
print(f"Percentage:  {total_lora:,.0f} / {total_model_params:,.0f} × 100 = {percentage:.3f}%")
print(f"\n→ Only {percentage:.3f}% of parameters are trainable!")
print(f"→ Full model: {total_model_params/1e9:.0f}B, LoRA: {total_lora/1e6:.1f}M")

## Exercise 2: DPO Loss by Hand

**Problem:** Given:
- $\log \pi_\theta(y_w|x) - \log \pi_{\text{ref}}(y_w|x) = 0.8$
- $\log \pi_\theta(y_l|x) - \log \pi_{\text{ref}}(y_l|x) = -0.4$
- $\beta = 0.1$

Calculate:
1. The **logit** $\beta(\text{ratio}_w - \text{ratio}_l)$
2. The **DPO loss** $-\log\sigma(\text{logit})$
3. The **gradient weight** $\sigma(\hat{r}_l - \hat{r}_w)$
4. Interpretation: is the model getting this pair right?

In [ ]:
# Exercise 2: Scaffold — fill in the ???

log_ratio_w = 0.8    # log π(yw|x) / πref(yw|x)
log_ratio_l = -0.4   # log π(yl|x) / πref(yl|x)
beta = 0.1

# Step 1: logit = β × (ratio_w - ratio_l)
logit = None       # ???

# Step 2: DPO loss = -log(σ(logit))
loss = None        # ???

# Step 3: gradient weight = σ(r̂_l - r̂_w) where r̂ = β × ratio
r_hat_w = None     # ???
r_hat_l = None     # ???
grad_weight = None # ???

print(f"Logit:        {logit}")
print(f"DPO Loss:     {loss:.4f}")
print(f"Grad weight:  {grad_weight:.4f}")
print(f"Model correct? {'Yes' if logit > 0 else 'No'}")

In [ ]:
# Exercise 2: SOLUTION

log_ratio_w = 0.8
log_ratio_l = -0.4
beta = 0.1

print("=== Exercise 2 Solution ===\n")

# Step 1
logit = beta * (log_ratio_w - log_ratio_l)
print(f"Step 1: logit = β × (ratio_w - ratio_l)")
print(f"        = {beta} × ({log_ratio_w} - ({log_ratio_l}))")
print(f"        = {beta} × {log_ratio_w - log_ratio_l}")
print(f"        = {logit}")

# Step 2
loss = -np.log(sigmoid(logit))
print(f"\nStep 2: σ(logit) = σ({logit}) = {sigmoid(logit):.4f}")
print(f"        loss = -log({sigmoid(logit):.4f}) = {loss:.4f}")

# Step 3
r_hat_w = beta * log_ratio_w
r_hat_l = beta * log_ratio_l
grad_weight = sigmoid(r_hat_l - r_hat_w)
print(f"\nStep 3: r̂_w = β × ratio_w = {beta} × {log_ratio_w} = {r_hat_w}")
print(f"        r̂_l = β × ratio_l = {beta} × {log_ratio_l} = {r_hat_l}")
print(f"        grad_weight = σ(r̂_l - r̂_w) = σ({r_hat_l} - {r_hat_w}) = σ({r_hat_l - r_hat_w}) = {grad_weight:.4f}")

print(f"\nStep 4: logit = {logit} > 0 → Model PREFERS y_w ✓")
print(f"        grad_weight = {grad_weight:.4f} < 0.5 → Model is mostly right, small gradient")
print(f"        → DPO will focus more on harder examples!")

## Exercise 3: QLoRA Memory Calculation

**Problem:** You want to QLoRA fine-tune a **70B** model with:
- $d = 8192$, $L = 80$ layers, $r = 32$
- LoRA on **Q and V** matrices only
- Base model in NF4 (4-bit), LoRA adapters in FP16

Calculate:
1. Base model memory (4-bit)
2. LoRA adapter memory (FP16)
3. Adam optimizer states memory
4. Total memory
5. Does it fit on a single A100-80GB?

In [ ]:
# Exercise 3: Scaffold — fill in the ???

N = 70e9
d = 8192
L = 80
r = 32
n_matrices = 2  # Q and V only

# Base model: 4-bit = 0.5 bytes per param
base_gb = None       # ??? = N * 0.5 / 1e9

# LoRA adapters: each matrix needs A(r×d) + B(d×r) in FP16 (2 bytes)
lora_params = None   # ??? = n_matrices * L * 2 * r * d
lora_gb = None       # ??? = lora_params * 2 / 1e9  (FP16)

# Adam states: 2 FP32 copies (momentum, variance) per LoRA param
optim_gb = None      # ??? = lora_params * 8 / 1e9

# Total
total_gb = None      # ???

print(f"Base model:  {base_gb:.1f} GB")
print(f"LoRA params: {lora_params/1e6:.1f}M → {lora_gb:.2f} GB")
print(f"Optimizer:   {optim_gb:.2f} GB")
print(f"Total:       {total_gb:.1f} GB")
print(f"Fits A100-80GB? {'Yes ✓' if total_gb < 80 else 'No ✗'}")

In [ ]:
# Exercise 3: SOLUTION

N = 70e9; d = 8192; L = 80; r = 32; n_matrices = 2

print("=== Exercise 3 Solution ===\n")

base_gb = N * 0.5 / 1e9
print(f"Base model:  {N/1e9:.0f}B × 0.5 bytes/param = {base_gb:.1f} GB")

lora_params = n_matrices * L * 2 * r * d
lora_gb = lora_params * 2 / 1e9
print(f"\nLoRA params: {n_matrices} mats × {L} layers × 2 × {r} × {d}")
print(f"           = {lora_params:,.0f} = {lora_params/1e6:.1f}M")
print(f"LoRA memory: {lora_params/1e6:.1f}M × 2 bytes = {lora_gb:.2f} GB")

optim_gb = lora_params * 8 / 1e9
print(f"\nOptimizer:   {lora_params/1e6:.1f}M × 8 bytes (Adam m+v in FP32) = {optim_gb:.2f} GB")

total_gb = base_gb + lora_gb + optim_gb
print(f"\nTotal = {base_gb:.1f} + {lora_gb:.2f} + {optim_gb:.2f} = {total_gb:.1f} GB")
print(f"\nFits A100-80GB? {'Yes ✓' if total_gb < 80 else 'No ✗'}")
print(f"(vs Full FT: {16*70:.0f} GB = {16*70/80:.0f}× A100s)")

## Exercise 4: EWC Loss Calculation

**Problem:** Given:
- $\theta = [1.0, 2.0]$, $\theta^* = [0.8, 2.3]$
- Fisher information $F = [10.0, 1.0]$
- $\lambda = 1.0$
- New task loss $\mathcal{L}_{\text{new}} = 0.5$

Calculate the **total EWC loss** $= \mathcal{L}_{\text{new}} + \frac{\lambda}{2}\sum_i F_i(\theta_i - \theta^*_i)^2$

Which parameter is penalised more and why?

In [ ]:
# Exercise 4: Scaffold — fill in the ???

theta = np.array([1.0, 2.0])
theta_star = np.array([0.8, 2.3])
F = np.array([10.0, 1.0])
lam = 1.0
L_new = 0.5

# EWC penalty for each parameter
ewc_term_0 = None  # ??? = (lam/2) * F[0] * (theta[0] - theta_star[0])**2
ewc_term_1 = None  # ??? = (lam/2) * F[1] * (theta[1] - theta_star[1])**2

# Total
ewc_penalty = None  # ??? = ewc_term_0 + ewc_term_1
total_loss = None   # ??? = L_new + ewc_penalty

print(f"EWC term 0: {ewc_term_0:.3f}")
print(f"EWC term 1: {ewc_term_1:.3f}")
print(f"EWC penalty: {ewc_penalty:.3f}")
print(f"Total loss:  {total_loss:.3f}")

In [ ]:
# Exercise 4: SOLUTION

theta = np.array([1.0, 2.0])
theta_star = np.array([0.8, 2.3])
F = np.array([10.0, 1.0])
lam = 1.0
L_new = 0.5

print("=== Exercise 4 Solution ===\n")

ewc_term_0 = (lam/2) * F[0] * (theta[0] - theta_star[0])**2
print(f"Term 0: (1/2) × {lam} × {F[0]} × ({theta[0]}-{theta_star[0]})² = 0.5 × {F[0]} × {(theta[0]-theta_star[0])**2} = {ewc_term_0:.3f}")

ewc_term_1 = (lam/2) * F[1] * (theta[1] - theta_star[1])**2
print(f"Term 1: (1/2) × {lam} × {F[1]} × ({theta[1]}-{theta_star[1]})² = 0.5 × {F[1]} × {(theta[1]-theta_star[1])**2} = {ewc_term_1:.3f}")

ewc_penalty = ewc_term_0 + ewc_term_1
total_loss = L_new + ewc_penalty

print(f"\nEWC penalty = {ewc_term_0:.3f} + {ewc_term_1:.3f} = {ewc_penalty:.3f}")
print(f"Total loss  = {L_new} + {ewc_penalty:.3f} = {total_loss:.3f}")

print(f"\nθ₀ penalty ({ewc_term_0:.3f}) >> θ₁ penalty ({ewc_term_1:.3f})")
print(f"Because F₀ = {F[0]} >> F₁ = {F[1]}")
print(f"→ θ₀ was IMPORTANT for the old task (high Fisher)")
print(f"→ EWC strongly penalises changes to θ₀, but lets θ₁ move freely")

## Exercise 5: Adapter Parameter Count

**Problem:** BERT-base with $d = 768$, $L = 12$ layers. Each layer gets a serial adapter with bottleneck $r = 64$.

Adapter module: $h \leftarrow h + f(hW_{\text{down}})W_{\text{up}}$, where:
- $W_{\text{down}} \in \mathbb{R}^{d \times r}$, $W_{\text{up}} \in \mathbb{R}^{r \times d}$
- Plus bias terms: $b_{\text{down}} \in \mathbb{R}^r$, $b_{\text{up}} \in \mathbb{R}^d$

**2 adapters per layer** (one after attention, one after FFN).
Total BERT-base params: 110M.

Calculate total adapter params and percentage.

In [ ]:
# Exercise 5: Scaffold — fill in the ???

d = 768
r = 64
L = 12
adapters_per_layer = 2
total_bert = 110e6

# Per adapter: W_down(d×r) + b_down(r) + W_up(r×d) + b_up(d)
params_per_adapter = None  # ???

# Per layer: 2 adapters
params_per_layer = None    # ???

# Total
total_adapter = None       # ???

# Percentage
pct = None                 # ???

print(f"Per adapter:    {params_per_adapter:,}")
print(f"Per layer:      {params_per_layer:,}")
print(f"Total adapters: {total_adapter:,}")
print(f"Percentage:     {pct:.2f}%")

In [ ]:
# Exercise 5: SOLUTION

d = 768; r = 64; L = 12; adapters_per_layer = 2; total_bert = 110e6

print("=== Exercise 5 Solution ===\n")

params_per_adapter = d * r + r + r * d + d
print(f"Per adapter: W_down({d}×{r}) + b_down({r}) + W_up({r}×{d}) + b_up({d})")
print(f"           = {d*r} + {r} + {r*d} + {d} = {params_per_adapter:,}")

params_per_layer = adapters_per_layer * params_per_adapter
print(f"\nPer layer:   {adapters_per_layer} × {params_per_adapter:,} = {params_per_layer:,}")

total_adapter = L * params_per_layer
print(f"Total:       {L} × {params_per_layer:,} = {total_adapter:,}")

pct = total_adapter / total_bert * 100
print(f"Percentage:  {total_adapter:,} / {total_bert/1e6:.0f}M = {pct:.2f}%")
print(f"\n→ Only {pct:.1f}% trainable params — adapters are very parameter-efficient!")

## Exercise 6: IFD Score

**Problem:** Compute the Instruction Following Difficulty score:
$$\text{IFD}(x, y) = \frac{\mathcal{L}(y|x)}{\mathcal{L}(y)}$$

Given: $\mathcal{L}(y|x) = 2.3$ (conditional loss), $\mathcal{L}(y) = 1.5$ (unconditional loss).

1. What is the IFD score?
2. Should this example be kept for training?
3. What if the losses were swapped?

In [ ]:
# Exercise 6: Scaffold — fill in the ???

L_cond = 2.3     # L(y|x) — loss given instruction
L_uncond = 1.5   # L(y) — loss without instruction

# IFD score
ifd = None       # ???

# Should we keep it?
keep = None      # ??? (True/False based on ifd < 1.0)

# Swapped case
ifd_swapped = None  # ??? = L_uncond / L_cond (what if instruction helps)

print(f"IFD = L(y|x) / L(y) = {L_cond}/{L_uncond} = {ifd:.3f}")
print(f"Keep? {keep}")
print(f"Swapped: IFD = {L_uncond}/{L_cond} = {ifd_swapped:.3f} → {'Keep' if ifd_swapped < 1 else 'Remove'}")

In [ ]:
# Exercise 6: SOLUTION

L_cond = 2.3; L_uncond = 1.5

print("=== Exercise 6 Solution ===\n")

ifd = L_cond / L_uncond
print(f"IFD = L(y|x) / L(y) = {L_cond} / {L_uncond} = {ifd:.3f}")
print(f"\nIFD = {ifd:.3f} > 1.0 → instruction HURTS prediction")
print(f"→ REMOVE this example — the instruction is misleading or contradictory")
print(f"→ The model does BETTER predicting y WITHOUT the instruction!")

print(f"\nSwapped case: IFD = {L_uncond}/{L_cond} = {L_uncond/L_cond:.3f}")
print(f"IFD = {L_uncond/L_cond:.3f} < 1.0 → instruction helps → KEEP ✓")

print(f"\nIntuition:")
print(f"  IFD < 1: instruction reduces uncertainty about y → good pair")
print(f"  IFD ≈ 1: instruction is irrelevant to y → marginal")
print(f"  IFD > 1: instruction increases uncertainty → noisy/conflicting pair")

## Exercise 7: GRPO Advantage Normalisation

**Problem:** Group size $G = 8$, rewards = $[1, 0, 1, 1, 0, 0, 1, 0]$ (binary: correct/incorrect)

Calculate:
1. Group mean $\mu$ and standard deviation $\sigma$
2. Normalised advantage $\hat{A}_i$ for each response
3. Which responses get positive/negative reinforcement?
4. What happens with all-correct rewards $[1,1,1,1,1,1,1,1]$?

In [ ]:
# Exercise 7: Scaffold — fill in the ???

rewards = np.array([1, 0, 1, 1, 0, 0, 1, 0], dtype=float)
G = len(rewards)

# Group statistics
mu = None       # ??? = np.mean(rewards)
sigma = None    # ??? = np.std(rewards)

# Normalised advantages
A_hat = None    # ??? = (rewards - mu) / (sigma + 1e-8)

print(f"μ = {mu:.4f}, σ = {sigma:.4f}")
print(f"\n{'i':<4} {'r_i':<6} {'Â_i':<10} {'Reinforcement'}")
print("-" * 30)
for i in range(G):
    sign = "+" if A_hat[i] > 0 else "-"
    print(f"{i:<4} {rewards[i]:<6.0f} {A_hat[i]:<+10.4f} {sign}")

# Edge case: all correct
rewards_all = np.ones(G)
sigma_all = np.std(rewards_all)
print(f"\nAll-correct: σ = {sigma_all}, → advantages = ???")

In [ ]:
# Exercise 7: SOLUTION

rewards = np.array([1, 0, 1, 1, 0, 0, 1, 0], dtype=float)
G = len(rewards)

print("=== Exercise 7 Solution ===\n")

mu = np.mean(rewards)
sigma = np.std(rewards)
A_hat = (rewards - mu) / (sigma + 1e-8)

print(f"μ = mean([1,0,1,1,0,0,1,0]) = {sum(rewards)}/{G} = {mu}")
print(f"σ = std = {sigma:.4f}\n")

print(f"{'i':<4} {'r_i':<6} {'r_i - μ':<10} {'Â_i':<12} {'Effect'}")
print("-" * 45)
for i in range(G):
    diff = rewards[i] - mu
    effect = "↑ reinforce" if A_hat[i] > 0 else "↓ discourage"
    print(f"{i:<4} {rewards[i]:<6.0f} {diff:<+10.2f} {A_hat[i]:<+12.4f} {effect}")

print(f"\nCorrect answers (r=1) get Â > 0 → reinforced")
print(f"Wrong answers (r=0) get Â < 0 → discouraged")
print(f"Sum of advantages = {np.sum(A_hat):.6f} ≈ 0 (zero-mean by construction)")

# Edge case
print(f"\n--- Edge case: all correct ---")
rewards_all = np.ones(G)
sigma_all = np.std(rewards_all)
print(f"σ = {sigma_all} → division by ~0 → advantages ≈ 0")
print(f"→ No gradient signal! All responses equal = nothing to learn from")
print(f"→ This is why GRPO needs DIVERSE outputs per prompt")

## Exercise 8: Model Merging with Task Vectors

**Problem:** Given:
- Base weights: $\theta_0 = [1.0, 1.0, 1.0]$
- Math fine-tuned: $\theta_1 = [1.2, 0.8, 1.5]$
- Code fine-tuned: $\theta_2 = [0.9, 1.1, 1.3]$

Calculate:
1. Task vectors $\tau_1 = \theta_1 - \theta_0$ and $\tau_2 = \theta_2 - \theta_0$
2. Simple average merged weights
3. Task vector addition: $\theta_{\text{merge}} = \theta_0 + \tau_1 + \tau_2$
4. Scaled task vectors with $\alpha = 0.5$: $\theta_{\text{merge}} = \theta_0 + 0.5(\tau_1 + \tau_2)$
5. Which method preserves the most from both tasks?

In [ ]:
# Exercise 8: Scaffold — fill in the ???

theta_0 = np.array([1.0, 1.0, 1.0])
theta_1 = np.array([1.2, 0.8, 1.5])  # math fine-tuned
theta_2 = np.array([0.9, 1.1, 1.3])  # code fine-tuned

# Task vectors
tau_1 = None   # ??? = theta_1 - theta_0
tau_2 = None   # ??? = theta_2 - theta_0

# Simple average
theta_avg = None  # ??? = (theta_1 + theta_2) / 2

# Task vector addition
theta_tv = None   # ??? = theta_0 + tau_1 + tau_2

# Scaled (α=0.5)
alpha = 0.5
theta_scaled = None  # ??? = theta_0 + alpha * (tau_1 + tau_2)

print(f"τ₁ (math):    {print_vector(tau_1)}")
print(f"τ₂ (code):    {print_vector(tau_2)}")
print(f"Simple avg:   {print_vector(theta_avg)}")
print(f"TV addition:  {print_vector(theta_tv)}")
print(f"TV scaled:    {print_vector(theta_scaled)}")

In [ ]:
# Exercise 8: SOLUTION

theta_0 = np.array([1.0, 1.0, 1.0])
theta_1 = np.array([1.2, 0.8, 1.5])
theta_2 = np.array([0.9, 1.1, 1.3])

print("=== Exercise 8 Solution ===\n")

# Task vectors
tau_1 = theta_1 - theta_0
tau_2 = theta_2 - theta_0
print(f"τ₁ = θ₁ - θ₀ = {print_vector(theta_1)} - {print_vector(theta_0)} = {print_vector(tau_1)}")
print(f"τ₂ = θ₂ - θ₀ = {print_vector(theta_2)} - {print_vector(theta_0)} = {print_vector(tau_2)}")

# Simple average
theta_avg = (theta_1 + theta_2) / 2
print(f"\n1. Simple average: ({print_vector(theta_1)} + {print_vector(theta_2)}) / 2")
print(f"   = {print_vector(theta_avg)}")

# Task vector addition
theta_tv = theta_0 + tau_1 + tau_2
print(f"\n2. TV addition: θ₀ + τ₁ + τ₂ = {print_vector(theta_0)} + {print_vector(tau_1)} + {print_vector(tau_2)}")
print(f"   = {print_vector(theta_tv)}")

# Scaled
alpha = 0.5
theta_scaled = theta_0 + alpha * (tau_1 + tau_2)
print(f"\n3. Scaled (α={alpha}): θ₀ + {alpha}(τ₁ + τ₂)")
print(f"   = {print_vector(theta_scaled)}")

# Analysis
print(f"\n--- Distance Analysis ---")
print(f"{'Method':<18} {'Dist to θ₁':<14} {'Dist to θ₂':<14} {'Dist to θ₀'}")
print("-" * 50)
for name, t in [("Simple avg", theta_avg), ("TV addition", theta_tv), ("TV scaled", theta_scaled)]:
    d1 = np.linalg.norm(t - theta_1)
    d2 = np.linalg.norm(t - theta_2)
    d0 = np.linalg.norm(t - theta_0)
    print(f"{name:<18} {d1:<14.4f} {d2:<14.4f} {d0:.4f}")

print(f"\n→ TV addition captures BOTH task changes but may overshoot")
print(f"→ Scaled TV (α=0.5) is more conservative, stays closer to θ₀")
print(f"→ Simple average loses the base model anchor")

print(f"\nNote: τ₁ and τ₂ conflict on dim 1: [{tau_1[1]:+.1f}] vs [{tau_2[1]:+.1f}]")
print(f"→ This is where TIES merging (sign election) would help!")